# Data and graph exploration

Descriptive statistics and visual sanity checks of the three graphs used by LLM-MGCL.
Nothing here is needed to reproduce the paper results - run `scripts/build_graphs.py` first,
and optionally train a model (`scripts/train.py --model llm_mgcl`) for the last section.

In [ ]:
import os, random, sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

from llm_mgcl.data import load_raw
from llm_mgcl.graphs.geo import haversine_to_all
from llm_mgcl.pipeline import load_dataset, load_trained_model, processed_files
from llm_mgcl.utils import get_device, load_config

cfg = load_config(os.environ.get("LLM_MGCL_CONFIG", "configs/default.yaml"))
device = get_device()
ds = load_dataset(cfg, device)
data, graphs = ds.data, ds.graphs
files = processed_files(cfg["paths"]["processed_dir"])
texts = pd.read_csv(files["item_texts"]).drop_duplicates("business_id").set_index("business_id")["text_combined"].to_dict()
_, business, _ = load_raw(cfg["paths"]["raw_dataset"])
coords = business.drop_duplicates("business_id").set_index("business_id")[["latitude", "longitude"]]

## Dataset statistics

In [ ]:
train = data.train_df
print(f"Users {data.n_users:,} | Items {data.n_items:,} | Train interactions {len(train):,}")
print(f"Density {len(train) / (data.n_users * data.n_items):.2e}")
print("Star distribution (train):")
print(train["stars"].value_counts().sort_index())

counts = data.item_train_counts()
print(counts.describe())
plt.figure(figsize=(8, 4))
plt.hist(counts, bins=50, orientation="horizontal")
plt.ylabel("Training interactions per item")
plt.xlabel(f"Number of items (total {len(counts):,})")
plt.title("Item popularity")
plt.show()

## G_UI - user-item sample

In [ ]:
rng = np.random.default_rng(42)
sample_users = rng.choice(train["user_id"].unique(), size=5, replace=False)
sub = train[train["user_id"].isin(sample_users)].groupby("user_id").head(3)

G = nx.Graph()
user_nodes = [f"U:{u[:6]}" for u in sub["user_id"].unique()]
item_nodes = [f"I:{i[:6]}" for i in sub["business_id"].unique()]
G.add_nodes_from(user_nodes); G.add_nodes_from(item_nodes)
G.add_edges_from((f"U:{r.user_id[:6]}", f"I:{r.business_id[:6]}", {"stars": r.stars}) for r in sub.itertuples())
pos = {n: (0, i) for i, n in enumerate(user_nodes)} | {n: (1, i) for i, n in enumerate(item_nodes)}

plt.figure(figsize=(10, 8))
nx.draw_networkx_nodes(G, pos, nodelist=user_nodes, node_size=800, node_color="lightblue")
nx.draw_networkx_nodes(G, pos, nodelist=item_nodes, node_size=600, node_color="lightgreen")
nx.draw_networkx_edges(G, pos, width=1.2)
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_edge_labels(G, pos, edge_labels={(u, v): f"{d['stars']:.0f}*" for u, v, d in G.edges(data=True)}, font_size=9)
plt.axis("off"); plt.title("User-item graph with star ratings (users blue, items green)"); plt.show()

## G_SEM - semantic neighbours

In [ ]:
sem_idx, sem_val = graphs.sem.indices().cpu(), graphs.sem.values().cpu()

def sem_neighbours(item_idx, topn=5):
    mask = sem_idx[0] == item_idx
    nbrs, wts = sem_idx[1][mask].tolist(), sem_val[mask].tolist()
    return sorted(zip(nbrs, wts), key=lambda x: -x[1])[:topn]

has_sem = torch.zeros(data.n_items, dtype=torch.bool)
has_sem[sem_idx[0].unique()] = True
print(f"Items with text: {len(texts):,} | items without any SEM edge: {(~has_sem).sum().item():,} of {data.n_items:,}")

for li in random.sample(torch.where(has_sem)[0].tolist(), 3):
    bid = data.idx2item[li]
    print("TARGET:", str(texts.get(bid, "<no text>"))[:160])
    for nb, w in sem_neighbours(li):
        print(f"   w={w:.3f}  {str(texts.get(data.idx2item[nb], '<no text>'))[:130]}")
    print()

In [ ]:
# Normalised edge weights of a random seed item and its neighbours
seed_item = random.choice(torch.where(has_sem)[0].tolist())
G = nx.Graph()
for nb, w in sem_neighbours(seed_item, topn=10):
    G.add_edge(data.idx2item[seed_item][:8], data.idx2item[nb][:8], weight=w)
pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(10, 8))
nx.draw(G, pos, with_labels=True, node_size=900, node_color="skyblue", font_size=7)
nx.draw_networkx_edge_labels(G, pos, edge_labels={(u, v): f"{d['weight']:.3f}" for u, v, d in G.edges(data=True)}, font_size=7)
plt.title("Item-item semantic graph (random sample, normalised weights)"); plt.axis("off"); plt.show()

In [ ]:
# Sanity checks: SEM edges must respect the distance filter; are items without SEM edges colder?
def edge_distances(idx, n=3000):
    sample = random.sample(range(idx.shape[1]), min(n, idx.shape[1]))
    a = [data.idx2item[idx[0, e].item()] for e in sample]
    b = [data.idx2item[idx[1, e].item()] for e in sample]
    la, lb = coords.reindex(a).to_numpy(), coords.reindex(b).to_numpy()
    return np.array([haversine_to_all(la[i:i+1, 0], la[i:i+1, 1], lb[i:i+1, 0], lb[i:i+1, 1])[0, 0] for i in range(len(a))])

d = edge_distances(sem_idx)
print(f"SEM edge distance (km): median={np.nanmedian(d):.1f}  max={np.nanmax(d):.1f}  "
      f">{cfg['graphs']['semantic']['max_dist_km']:.0f} km: {np.mean(d > cfg['graphs']['semantic']['max_dist_km']) * 100:.1f}%")
for name, mask in [("with SEM edges", has_sem), ("without SEM edges", ~has_sem)]:
    c = counts.reindex([data.idx2item[i] for i in torch.where(mask)[0].tolist()]).fillna(0)
    print(f"{name:<18}: median interactions {c.median():.0f} | share cold (<=20) {np.mean(c <= 20) * 100:.1f}%")

## G_GEO - geographic neighbours

In [ ]:
geo_idx = graphs.geo.indices().cpu()
item = random.choice(geo_idx[0].unique().tolist())
nbrs = geo_idx[1][geo_idx[0] == item].tolist()
bid = data.idx2item[item]
lat0, lon0 = coords.loc[bid]
pts = coords.reindex([data.idx2item[n] for n in nbrs])
dist = haversine_to_all(np.array([lat0]), np.array([lon0]), pts["latitude"].values, pts["longitude"].values)[0]
order = np.argsort(dist)

fig, ax = plt.subplots(figsize=(10, 8))
for rank, j in enumerate(order, 1):
    la, lo = pts.iloc[j]
    ax.scatter(lo, la, c="steelblue", s=120, alpha=0.7, edgecolor="black", linewidth=0.5, zorder=3)
    ax.annotate(f"#{rank}\n{dist[j]:.2f} km", xy=(lo, la), xytext=(5, 5), textcoords="offset points", fontsize=8)
    ax.plot([lon0, lo], [lat0, la], "gray", linestyle="--", alpha=0.4, zorder=1)
ax.scatter(lon0, lat0, c="crimson", s=300, marker="*", edgecolor="black", zorder=5, label="target item")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"Geographic neighbours of {bid[:12]}..."); plt.show()

d = edge_distances(geo_idx)
print(f"GEO edge distance (km): median={np.nanmedian(d):.2f}  max={np.nanmax(d):.2f}")

## Example recommendations (requires a trained LLM-MGCL checkpoint)

In [ ]:
model = load_trained_model("llm_mgcl", cfg["training"]["seeds"][0], cfg, data, device)
if model is None:
    print("No checkpoint found - run scripts/train.py --model llm_mgcl first.")
else:
    views = {k: v for k, v in model.view_embeddings(graphs).items()}
    views["full"] = views["cf"] + views["sem"] + views["geo"]

    @torch.no_grad()
    def recommend(user_id, view="full", k=10):
        scores = views[view] @ views["user"][data.user2idx[user_id]]
        seen = [data.item2idx[i] for i in data.train_pos.get(user_id, set())]
        scores[torch.tensor(seen, device=scores.device)] = -1e9
        return [data.idx2item[i] for i in torch.topk(scores, k).indices.tolist()]

    user = random.choice([u for u in data.test_pos if u in data.user2idx])
    print("Visited (train):")
    for b in list(data.train_pos[user])[:5]:
        print("   ", str(texts.get(b, "<no text>"))[:110])
    for view in ["full", "cf", "sem"]:
        print(f"\nTop-5 via '{view}' view:")
        for b in recommend(user, view, 5):
            print("   ", str(texts.get(b, "<no text>"))[:110])